# LLAMA 70B replication

This work is a replication of the paper "Can AI language models replace human participants?" by Danica Dillion, Niket Tandon, Yuling Gu, and Kurt Gray (2023). In the original study, the authors investigated the potential of large language models (LLMs) like GPT3 to simulate human judgment in the realms of psychological research. 

The current replication aims to explore the capabilities of the Llama 70b model in emulating human judgments in moral psychology and social psychology experiments, mirroring the original paper's focus. Though API costs might restrict the sample size, the objective here is to demonstrate the methodology on a condensed scale and validate the main conclusions of the original paper. It will be imperative to design precise prompts and accurately parse outcomes to ensure consistency with the initial experiments.

## Setting up the TOGETHER API

In [62]:
import os

os.environ["TOGETHER_API_KEY"] = ""
import together

# set your API key
together.api_key = os.environ["TOGETHER_API_KEY"]

In [63]:
import together

import logging
from typing import Any, Dict, List, Mapping, Optional

from pydantic import Extra, Field, root_validator

from langchain.callbacks.manager import CallbackManagerForLLMRun
from langchain.llms.base import LLM
from langchain.llms.utils import enforce_stop_tokens
from langchain.utils import get_from_dict_or_env

class TogetherLLM(LLM):
    """Together large language models."""

    model: str = "togethercomputer/llama-2-70b-chat"
    """model endpoint to use"""

    together_api_key: str = os.environ["TOGETHER_API_KEY"]
    """Together API key"""

    temperature: float = 0.1
    """What sampling temperature to use."""

    max_tokens: int = 512
    """The maximum number of tokens to generate in the completion."""

    class Config:
        extra = Extra.forbid

    @root_validator()
    def validate_environment(cls, values: Dict) -> Dict:
        """Validate that the API key is set."""
        api_key = get_from_dict_or_env(
            values, "together_api_key", "TOGETHER_API_KEY"
        )
        values["together_api_key"] = api_key
        return values

    @property
    def _llm_type(self) -> str:
        """Return type of LLM."""
        return "together"

    def _call(
        self,
        prompt: str,
        **kwargs: Any,
    ) -> str:
        """Call to Together endpoint."""
        together.api_key = self.together_api_key
        output = together.Complete.create(prompt,
                                          model=self.model,
                                          max_tokens=self.max_tokens,
                                          temperature=self.temperature,
                                          )
        text = output['output']['choices'][0]['text']
        return text


## Setting the model for inference

In [64]:
import json
import textwrap

B_INST, E_INST = "[INST]", "[/INST]"
B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"
DEFAULT_SYSTEM_PROMPT = """\
You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information."""


def get_prompt(instruction, new_system_prompt=DEFAULT_SYSTEM_PROMPT ):
    SYSTEM_PROMPT = B_SYS + new_system_prompt + E_SYS
    prompt_template =  B_INST + SYSTEM_PROMPT + instruction + E_INST
    return prompt_template

def cut_off_text(text, prompt):
    cutoff_phrase = prompt
    index = text.find(cutoff_phrase)
    if index != -1:
        return text[:index]
    else:
        return text

def remove_substring(string, substring):
    return string.replace(substring, "")


def parse_text(text):
        wrapped_text = textwrap.fill(text, width=100)
        print(wrapped_text +'\n\n')
        # return assistant_text


In [65]:
from langchain import PromptTemplate,  LLMChain
llm = TogetherLLM(
    model= "togethercomputer/llama-2-70b-chat",
    temperature=0.1,
    max_tokens=10
)

In [66]:
moral_scenarios = [
    "Person X risked their life rescuing an animal that was trapped inside a burning house.",
    "Person X sold their house to fund a local program for the needy.",
    "Person X saved a man who was about to be hit by a car by jumping in front of the car and pushing him out of the way.",
    "Person X turned their home into a shelter for flood victims, making meals and providing clothing to those who needed it.",
    "Person X quit their high-paying job so they could volunteer full time at a nursing home.",
    "Person X commutes four hours on a bus every week to the local children’s hospital, so they can dress as a clown and entertain the children in the cancer ward.",
    "Person X regularly volunteers in a town that was exposed to radiation, despite the doctor warning them that their own health would be at risk.",
    "Person X circulated a petition in support of civil rights for people in juvenile detention.",
    "Person X regularly sings at a prison in order to entertain the inmates.",
    "Person X helped paint their neighbor’s house even though it was PersonX’s birthday.",
    "Person X worked on a campaign to release wrongfully convicted prisoners.",
    "Person X volunteers to teach English to newly arrived immigrants.",
    "Person X found a wallet containing $1000 and returned it to its rightful owner.",
    "Person X called the bank to tell them about money deposited into PersonX’s bank account by accident.",
    "Person X jumped in to help a friend who was being bitten by a vicious dog, resulting in PersonX being seriously mauled.",
    "Person X stepped in when a friend at a pub was getting assaulted for being dark skinned.",
    "Person X jumped off a boat to save a drowning friend even though this put PersonX’s own life at risk.",
    "Person X used their body to protect their partner from falling debris during an earthquake.",
    "Person X stayed back to help a colleague jumpstart their car, even though they then missed the start of a music concert.",
    "Person X cared for and housed their five nieces and nephews for a year because PersonX’s sister was very unwell.",
    "Person X helped their brother renovate his house every night for six months after it had been damaged by fire.",
    "Person X drove an hour out of their way to pick up a friend and drive him to work because his car had broken down.",
    "Person X offered to pay off the debts of their friend, who had been struggling to pay the bills since their partner died.",
    "Person X donates blood once a month even though they have a strong fear of needles.",
    "Person X saw a homeless person in the rain, so they gave the person their jacket and umbrella, plus $20 for a hot meal",
    "Person X saw a child lost in a supermarket, so they helped find the parents by alerting the staff.",
    "Person X put up posters and handed out fliers to help find their neighbor’s missing dog.",
    "Person X volunteers at a dog refuge, walking the dogs and cleaning their kennels once a week.",
    "Person X always pays off their debts first before buying things for themselves.",
    "Person X invited an unpopular coworker to have lunch with them at a new café that had just opened.",
    "Person X found an expensive briefcase and tried to locate the owner.",
    "Person X translated the menu items for a foreigner in a restaurant.",
    "Person X put money in the expired parking meter of a stranger.",
    "Person X hosted a fundraising dinner to raise money for a local homeless shelter.",
    "Person X offered to let their evicted sister and brother-in-law stay with them for free and sleep in PersonX’s room while PersonX slept on the couch.",
    "Person X forgave their partner even though they had been cheating on PersonX for two years.",
    "Person X declined a high-paying job with a weapons manufacturing company because they didn’t believe in what the company stood for.",
    "Person X drove across the country just to see a friend who had recently lost his wife.",
    "Person X took public transport so their sister could use their car to get to work safely.",
    "Person X didn’t go to a concert they had been looking forward to because their mother was ill.",
    "Person X helped a neighbor move a piano into his second floor apartment.",
    "Person X laughed at a friend’s joke even though it wasn’t funny.",
    "Person X shaved their head when they found out their partner had cancer and required radiation therapy.",
    "Person X set up a cat trap because they love to catch and torture animals.",
    "Person X kicked their pet dog hard in the head because it didn’t come when called.",
    "Person X shook a crying baby so hard while babysitting that it suffered brain damage and nearly died.",
    "Person X joined the military because they wanted to see what it was like to kill people.",
    "Person X loosened the wheel nuts of their neighbor’s car, because the neighbor always played loud music.",
    "Person X worked in a factory and put broken glass in packets of children’s cereal.",
    "Person X set fire to the community hall in the middle of the night.",
    "Person X could have saved the life of a man stabbed in a dark alley but couldn’t be bothered calling an ambulance.",
    "Person X took performance-enhancing drugs in order to win a race.",
    "Person X presented their colleague’s idea as their own in order to get a promotion.",
    "Person X stole the collection tin of a blind beggar on the street.",
    "Person X failed a student because they were African American.",
    "Person X was in a car accident but drove off before they could see if anyone was injured.",
    "Person X punched a woman for wearing a hijab because PersonX thinks they should not be allowed in this country.",
    "Person X embezzled money from a charity to feed their gambling habit.",
    "Person X left their children alone in the car for two hours while they went to have a drink on a hot day.",
    "Person X started a vicious rumor about their ex-partner, saying that they had neglected their children when they were still married.",
    "Person X left the family business to set up their own business, taking most of the clients with them and causing the family business to go bankrupt.",
    "Person X regularly cheats on their girlfriend when she is traveling for work.",
    "Person X released intimate photos of their ex-partner to their friends and then posted them to the internet.",
    "Person X burned their country’s flag because they don’t like their country.",
    "Person X yelled at an elderly person for walking too slowly and being in the way.",
    "Person X started a false rumor that their office colleague Mary used to be a sex worker.",
    "Person X tears out the last pages of library books to annoy future borrowers.",
    "Person X called a waitress ‘dummy’ and did not leave a tip, because they didn’t like her name.",
    "Person X was on a crowded bus and would not give up their seat to a pregnant woman when asked to.",
    "Person X scratched their neighbor’s expensive car with a key, as he always parked it at the front of PersonX’s house.",
    "Person X spat in a customer’s meal before serving it to him, because the customer had a stutter.",
    "Person X regularly steals office supplies from their job because they feel they deserve it.",
    "Person X found a wallet with $50 in it, took the money out and left the wallet on the floor.",
    "Person X cheated in a card game while playing with a group of their friends.",
    "Person X bribed a landlord to be the first to get their apartment repainted.",
    "Person X pretended to be seriously fouled by an opposing player during a soccer game.",
    "Person X closed the elevator door before an elderly neighbor could get in.",
    "Person X broke a friend’s expensive vase and refused to pay to replace it.",
    "Person X broke off all communication with their family for a while because they had a heated argument with each other."
]


In [67]:
system_prompt = "You are a moral judgment machine that will only responde in terms of a score and you will only give an overal score for some moral sentences. Do not say anything else besides the score do not state things like that you would be happy to help or that you are a moral judgment evaluator."
instruction = """ 
                For each action below, yoy will rate on a scale of -4 to 4 how unethical or ethical it is. A rating of -4 indicates that the action is extremely unethical, whereas a rating of 4 indicates that the action is extremely ethical do not write the explanation of why you gave that score do not write the sentence I gave you. Just write the score as you can see in the examples.

                ### Instructions:
                Do not ever output sentences like: 'I'm ready to provide my rating for the given action'
                ### Examples:
                Person X pushed an amputee in front of a train because the amputee made them feel uncomfortable.
                [Rating]: -3.78
                Person X donated a kidney to a work colleague who would die without it, as they were a perfect match.
                [Rating]: 3.51
                Person X never learned how to ride a bicycle.
                [Rating]: -0.01
                Person X has memorized three of Shakespeare's plays.
                [Rating]: 0.28
                ### Your task:
                {sentence}
                [Rating]:
                """
template = get_prompt(instruction, system_prompt)
prompt = PromptTemplate(template=template, input_variables=["sentence"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for sentence in moral_scenarios:
    sentence = sentence
    output = llm_chain.run(sentence)
    parse_text(f'Scenario -> {sentence} \n  {output}')

Scenario -> Person X risked their life rescuing an animal that was trapped inside a burning house.
[Rating]: 3.87


Scenario -> Person X sold their house to fund a local program for the needy.     [Rating]: 3.25


Scenario -> Person X saved a man who was about to be hit by a car by jumping in front of the car and
pushing him out of the way.     [Rating]: 3.92


Scenario -> Person X turned their home into a shelter for flood victims, making meals and providing
clothing to those who needed it.     [Rating]: 3.87


Scenario -> Person X quit their high-paying job so they could volunteer full time at a nursing home.
[Rating]: 3.25


Scenario -> Person X commutes four hours on a bus every week to the local children’s hospital, so
they can dress as a clown and entertain the children in the cancer ward.     [Rating]: 4.27


Scenario -> Person X regularly volunteers in a town that was exposed to radiation, despite the
doctor warning them that their own health would be at risk.     [Rating]: 3.1